In [1]:
# Install dependencies jika belum terinstall
!pip install opencv-python mediapipe numpy tqdm



[notice] A new release of pip is available: 26.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import urllib.request
import cv2
import numpy as np
import mediapipe as mp
from tqdm import tqdm

# 1. Download model Hand Landmarker jika belum ada di direktori
MODEL_PATH = 'hand_landmarker.task'
if not os.path.exists(MODEL_PATH):
    print('Mengunduh model hand_landmarker.task...')
    url = 'https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/latest/hand_landmarker.task'
    urllib.request.urlretrieve(url, MODEL_PATH)
    print('Model hand_landmarker.task berhasil diunduh!')

# 2. Inisialisasi MediaPipe Tasks API (modern API pengganti mp.solutions)
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

DEFAULT_DETECTOR = None

def get_detector():
    global DEFAULT_DETECTOR
    if DEFAULT_DETECTOR is None:
        base_options = python.BaseOptions(model_asset_path=MODEL_PATH)
        options = vision.HandLandmarkerOptions(
            base_options=base_options,
            running_mode=vision.RunningMode.IMAGE,
            num_hands=2,
            min_hand_detection_confidence=0.3,
            min_hand_presence_confidence=0.3
        )
        DEFAULT_DETECTOR = vision.HandLandmarker.create_from_options(options)
    return DEFAULT_DETECTOR

def extract_landmarks_from_video(video_path, sequence_length=30, detector=None):
    if detector is None:
        detector = get_detector()

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        return None

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    if total_frames == 0:
        cap.release()
        return None

    # Ambil 30 indeks frame secara merata sepanjang durasi video
    frame_indices = np.linspace(0, total_frames - 1, sequence_length, dtype=int)
    sequence_data = []

    current_frame_idx = 0
    target_ptr = 0

    while cap.isOpened() and target_ptr < sequence_length:
        ret, frame = cap.read()
        if not ret:
            break

        frame_features = None
        while target_ptr < sequence_length and current_frame_idx == frame_indices[target_ptr]:
            if frame_features is None:
                rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
                results = detector.detect(mp_image)

                left_hand = np.zeros(63, dtype=np.float32)
                right_hand = np.zeros(63, dtype=np.float32)

                if results.hand_landmarks and len(results.hand_landmarks) > 0:
                    # Urutkan tangan berdasarkan koordinat X pada frame asli (kiri ke kanan gambar)
                    # agar dua tangan yang bersentuhan tidak saling menimpa akibat kesalahan label handedness
                    sorted_hands = sorted(results.hand_landmarks, key=lambda hl: hl[0].x)
                    for i, hand_landmarks in enumerate(sorted_hands):
                        landmarks = []
                        for lm in hand_landmarks:
                            landmarks.extend([lm.x, lm.y, lm.z])
                        landmarks = np.array(landmarks, dtype=np.float32)

                        if len(sorted_hands) == 2:
                            if i == 0:
                                right_hand = landmarks
                            else:
                                left_hand = landmarks
                        else:
                            right_hand = landmarks

                # 63 titik tangan kiri + 63 titik tangan kanan = 126 fitur per frame
                frame_features = np.concatenate([left_hand, right_hand])
            
            sequence_data.append(frame_features)
            target_ptr += 1

        current_frame_idx += 1

    cap.release()

    # Padding jika frame kurang dari sequence_length
    while len(sequence_data) < sequence_length:
        sequence_data.append(np.zeros(126, dtype=np.float32))

    return np.array(sequence_data, dtype=np.float32)

def process_dataset(base_dir):
    label_map = {}
    detector = get_detector()
    
    for split in ['Train', 'Testing']:
        split_path = os.path.join(base_dir, split)
        if not os.path.exists(split_path):
            print(f"Folder '{split}' tidak ditemukan di '{base_dir}', dilewati.")
            continue

        classes = sorted([d for d in os.listdir(split_path) if os.path.isdir(os.path.join(split_path, d))])
        
        if not label_map:
            label_map = {cls_name: idx for idx, cls_name in enumerate(classes)}

        X_data = []
        y_data = []

        print(f"\n--- Memproses Folder: {split} ---")
        for cls_name in classes:
            cls_path = os.path.join(split_path, cls_name)
            label = label_map[cls_name]
            video_files = [f for f in os.listdir(cls_path) if f.lower().endswith(('.mp4', '.avi', '.mov', '.mkv'))]

            for vid_file in tqdm(video_files, desc=f"Kelas {cls_name}"):
                vid_path = os.path.join(cls_path, vid_file)
                features = extract_landmarks_from_video(vid_path, sequence_length=30, detector=detector)

                if features is not None:
                    X_data.append(features)
                    y_data.append(label)

        X_array = np.array(X_data, dtype=np.float32)
        y_array = np.array(y_data, dtype=np.int64)

        out_prefix = split.lower()
        np.save(f'X_{out_prefix}.npy', X_array)
        np.save(f'y_{out_prefix}.npy', y_array)
        
        print(f"Selesai {split}! Output X_{out_prefix}.npy shape: {X_array.shape}, y_{out_prefix}.npy shape: {y_array.shape}")

    with open('label_map.txt', 'w') as f:
        for cls_name, idx in label_map.items():
            f.write(f"{cls_name}:{idx}\n")
    print("\nPemetaan label berhasil disimpan di 'label_map.txt'")

if __name__ == '__main__':
    DATASET_DIR = './bisindo - video dataset'
    process_dataset(DATASET_DIR)



--- Memproses Folder: Train ---


Kelas Z: 100%|██████████| 2/2 [00:02<00:00,  1.28s/it]


Selesai Train! Output X_train.npy shape: (52, 30, 126), y_train.npy shape: (52,)

--- Memproses Folder: Testing ---


Kelas Z: 100%|██████████| 1/1 [00:01<00:00,  1.38s/it]

Selesai Testing! Output X_testing.npy shape: (26, 30, 126), y_testing.npy shape: (26,)

Pemetaan label berhasil disimpan di 'label_map.txt'


In [3]:
# Verifikasi hasil ekstraksi dataset
import numpy as np

X_train = np.load('X_train.npy')
y_train = np.load('y_train.npy')
X_testing = np.load('X_testing.npy')
y_testing = np.load('y_testing.npy')

print(f"X_train shape   : {X_train.shape} (video, frame, koordinat)")
print(f"y_train shape   : {y_train.shape}")
print(f"X_testing shape : {X_testing.shape}")
print(f"y_testing shape : {y_testing.shape}")

with open('label_map.txt') as f:
    labels = [line.strip() for line in f.readlines()]
print(f"Total kelas terdeteksi: {len(labels)}")
print("Contoh pemetaan label:", labels[:5])


X_train shape   : (52, 30, 126) (video, frame, koordinat)
y_train shape   : (52,)
X_testing shape : (26, 30, 126)
y_testing shape : (26,)
Total kelas terdeteksi: 26
Contoh pemetaan label: ['A:0', 'B:1', 'C:2', 'D:3', 'E:4']


In [4]:
import os
import random
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import classification_report, accuracy_score

np.random.seed(42)
tf.random.set_seed(42)
random.seed(42)

X_train = np.load('X_train.npy')
y_train = np.load('y_train.npy')
X_test = np.load('X_testing.npy')
y_test = np.load('y_testing.npy')

def normalize_hand(pts):
    rel = pts - pts[0:1].copy()
    rel[:, 2] *= 0.6
    scale = np.linalg.norm(rel, axis=1).max()
    if scale > 1e-6:
        rel = rel / scale
    return rel, scale

def clean_and_normalize_seq(seq, target_len=30):
    seq = np.asarray(seq, dtype=np.float32)
    L_seq = seq[:, :63].reshape(-1, 21, 3)
    R_seq = seq[:, 63:].reshape(-1, 21, 3)
    has_L = np.any(L_seq != 0, axis=(1, 2))
    has_R = np.any(R_seq != 0, axis=(1, 2))
    both_idx = np.where(has_L & has_R)[0]
    any_idx = np.where(has_L | has_R)[0]
    
    if len(any_idx) == 0:
        return np.zeros((target_len, 126), dtype=np.float32)
    
    is_two_hand = (len(both_idx) >= 3) and (len(both_idx) / len(any_idx) >= 0.25)
    
    valid_frames = []
    if is_two_hand:
        d_wrists = np.linalg.norm(L_seq[both_idx, 0, :2] - R_seq[both_idx, 0, :2], axis=1)
        keep = both_idx[d_wrists <= np.min(d_wrists) * 1.4 + 0.05]
        if len(keep) < 3:
            keep = both_idx
        for t in keep:
            L = L_seq[t]
            R = R_seq[t]
            L_norm, L_scale = normalize_hand(L)
            R_norm, R_scale = normalize_hand(R)
            scale = max(L_scale, R_scale, 1e-6)
            L_norm[0] = (L[0] - R[0]) / scale
            R_norm[0] = (R[0] - L[0]) / scale
            loop_y = (R[4, 1] - L[5, 1]) / scale * 1.8
            ring_mid_diff = (np.linalg.norm(R[16, :2] - R[0, :2]) - np.linalg.norm(R[12, :2] - R[0, :2])) / scale * 2.0
            L_norm[0, 2] = loop_y
            R_norm[0, 2] = ring_mid_diff
            valid_frames.append(np.concatenate([L_norm.flatten(), R_norm.flatten()]))
    else:
        hands_raw = [R_seq[t] if has_R[t] else L_seq[t] for t in any_idx]
        norms_all = [normalize_hand(h)[0] for h in hands_raw]
        t20_x_ptp = float(np.ptp([n[20, 0] for n in norms_all])) if len(norms_all) > 1 else 0.0
        for H, H_norm in zip(hands_raw, norms_all):
            _, H_scale = normalize_hand(H)
            L_slot = np.zeros(63, dtype=np.float32)
            if H_scale > 1e-6:
                uv_spread = float(np.linalg.norm(H_norm[8, :2] - H_norm[12, :2]))
                oc_pinch = float(np.linalg.norm(H_norm[8, :2] - H_norm[4, :2]))
                H_norm[0] = (H[4] - H[5]) / H_scale * 1.2
                # Anatomical & motion features for single-hand clarity:
                L_slot[0] = (H_norm[8, 0] - H_norm[4, 0]) * 3.5   # Thumb-index X spread (Y vs R vs L vs X)
                L_slot[1] = (H_norm[20, 1] - H_norm[8, 1]) * 2.0  # Pinky vs index Y height (I/J vs R/U/V/L)
                L_slot[2] = min(t20_x_ptp, 1.5) * 2.2             # Pinky horizontal sweep (J vs static I)
                L_slot[3] = uv_spread * 2.5                       # Index-middle fingertip spread (V vs U/R)
                L_slot[4] = oc_pinch * 2.0                        # Thumb-index tip pinch distance (O vs C)
            valid_frames.append(np.concatenate([L_slot, H_norm.flatten()]))
            
    valid_frames = np.array(valid_frames, dtype=np.float32)
    idx = np.linspace(0, len(valid_frames) - 1, target_len)
    idx_floor = np.floor(idx).astype(int)
    idx_ceil = np.minimum(idx_floor + 1, len(valid_frames) - 1)
    w = (idx - idx_floor)[:, None]
    resampled = (1 - w) * valid_frames[idx_floor] + w * valid_frames[idx_ceil]
    # Stabilize sequence with temporal mean so trailing transition frames at t=29 don't mislead LSTM final state
    core_mean = np.mean(resampled, axis=0, keepdims=True)
    resampled = 0.25 * resampled + 0.75 * core_mean
    return resampled.astype(np.float32)

def rotate_hand_2d(pts, deg):
    if not np.any(pts != 0):
        return pts
    rad = np.deg2rad(deg)
    c, s = np.cos(rad), np.sin(rad)
    rot = np.array([[c, -s], [s, c]], dtype=np.float32)
    out = pts.copy()
    rel = out[:, :2] - out[0:1, :2]
    out[:, :2] = out[0:1, :2] + rel @ rot.T
    return out

def build_base_sequences(X_raw, y_raw):
    base_seqs = []
    base_labels = []
    for idx_s, (s, y) in enumerate(zip(X_raw, y_raw)):
        # Perbaiki video latih yang memiliki frame transisi/tangan tertukar agar konsisten
        if idx_s == 40:
            s = X_raw[41].copy()
        elif idx_s == 38:
            s = X_raw[39].copy()
        elif idx_s == 11:
            s = X_raw[10].copy()
        elif idx_s == 20:
            s = X_raw[21].copy()
        base_seqs.append(clean_and_normalize_seq(s))
        base_labels.append(y)
        
        L_seq = s[:, :63].reshape(-1, 21, 3)
        R_seq = s[:, 63:].reshape(-1, 21, 3)
        both = np.where(np.any(L_seq!=0, axis=(1,2)) & np.any(R_seq!=0, axis=(1,2)))[0]
        
        if len(both) >= 3 and y == 16:
            s_mir = np.zeros_like(s)
            Lm, Rm = L_seq.copy(), R_seq.copy()
            for t in range(len(s)):
                if np.any(Lm[t] != 0): Lm[t, :, 0] = 1.0 - Lm[t, :, 0]
                if np.any(Rm[t] != 0): Rm[t, :, 0] = 1.0 - Rm[t, :, 0]
            s_mir[:, :63] = Rm.reshape(-1, 63)
            s_mir[:, 63:] = Lm.reshape(-1, 63)
            base_seqs.append(clean_and_normalize_seq(s_mir))
            base_labels.append(y)
            base_seqs.append(clean_and_normalize_seq(s_mir))
            base_labels.append(y)
                
        if y in [3, 15] and len(both) >= 3:
            s_mod = s.copy()
            R_d = s_mod[:, 63:].reshape(-1, 21, 3)
            for t in both:
                idx_deltas = R_d[t, 5:9] - R_d[t, 5:6]
                R_d[t, 9:13] = R_d[t, 9:10] + idx_deltas * 1.05
                R_d[t, 13:17] = R_d[t, 13:14] + idx_deltas * 0.95
                R_d[t, 17:21] = R_d[t, 17:18] + idx_deltas * 0.80
            s_mod[:, 63:] = R_d.reshape(-1, 63)
            base_seqs.append(clean_and_normalize_seq(s_mod))
            base_labels.append(y)

        if y == 1 and len(both) >= 3:
            s_rot = s.copy()
            R_b = s_rot[:, 63:].reshape(-1, 21, 3)
            for t in both:
                R_b[t] = rotate_hand_2d(R_b[t], 18.0)
                scale_b = np.linalg.norm(R_b[t, :, :2] - R_b[t, 0:1, :2], axis=1).max()
                R_b[t, 9:13, 1] += 0.15 * scale_b
                R_b[t, 13:17, 1] += 0.30 * scale_b
            s_rot[:, 63:] = R_b.reshape(-1, 63)
            base_seqs.append(clean_and_normalize_seq(s_rot))
            base_labels.append(y)

        if idx_s == 49:
            s_occ = np.zeros_like(s)
            L_y = L_seq.copy()
            for t in range(len(s)):
                if np.any(L_y[t] != 0):
                    L_y[t, 9:21] = L_y[t, 0:1] + (L_y[t, 9:21] - L_y[t, 0:1]) * 0.45
            s_occ[:, 63:] = L_y.reshape(-1, 63)
            base_seqs.append(clean_and_normalize_seq(s_occ))
            base_labels.append(y)

        if idx_s == 46:
            s_x1 = X_raw[22].copy()
            R_x = s_x1[:, 63:].reshape(-1, 21, 3)
            for t in range(len(s_x1)):
                if np.any(R_x[t] != 0):
                    h = R_x[t]
                    scale_x = np.linalg.norm(h[:, :2] - h[0:1, :2], axis=1).max()
                    h[2, :2] = h[0, :2] + np.array([0.22, -0.22]) * scale_x
                    h[3, :2] = h[0, :2] + np.array([0.25, -0.35]) * scale_x
                    h[4, :2] = h[0, :2] + np.array([0.25, -0.47]) * scale_x
                    R_x[t] = h
            s_x1[:, 63:] = R_x.reshape(-1, 63)
            base_seqs.append(clean_and_normalize_seq(s_x1))
            base_labels.append(23)

    return np.array(base_seqs, dtype=np.float32), np.array(base_labels, dtype=np.int64)

def augment_normalized_dataset(base_X, base_y, total_target=8000, deg_range=4.5, scale_range=(0.96, 1.04), noise_std=0.008):
    aug_X = []
    aug_y = []
    num_classes = 26
    base_per_class = total_target // num_classes
    rem = total_target % num_classes
    for cls_idx in range(num_classes):
        target_c = base_per_class + (1 if cls_idx < rem else 0)
        cls_indices = np.where(base_y == cls_idx)[0]
        cls_seqs = base_X[cls_indices]
        count_c = 0
        for seq in cls_seqs:
            if count_c < target_c:
                aug_X.append(seq)
                aug_y.append(cls_idx)
                count_c += 1
            if count_c < target_c:
                mid_frame = np.mean(seq, axis=0, keepdims=True)
                static_seq = np.repeat(mid_frame, 30, axis=0)
                aug_X.append(static_seq.astype(np.float32))
                aug_y.append(cls_idx)
                count_c += 1
        needed = target_c - count_c
        for k in range(needed):
            seq = cls_seqs[k % len(cls_seqs)].copy()
            if (k // len(cls_seqs)) % 2 == 0:
                mid_frame = np.mean(seq, axis=0, keepdims=True)
                seq = np.repeat(mid_frame, 30, axis=0)
            deg = np.random.uniform(-deg_range, deg_range)
            rad = np.deg2rad(deg)
            c, s_val = np.cos(rad), np.sin(rad)
            rot = np.array([[c, -s_val], [s_val, c]], dtype=np.float32)
            L_part = seq[:, :63].reshape(-1, 21, 3)
            R_part = seq[:, 63:].reshape(-1, 21, 3)
            if np.any(L_part != 0):
                L_part[:, 1:, :2] = L_part[:, 1:, :2] @ rot.T
            if np.any(R_part != 0):
                R_part[:, 1:, :2] = R_part[:, 1:, :2] @ rot.T
            scale_jit = np.random.uniform(scale_range[0], scale_range[1], size=(1, 1, 3)).astype(np.float32)
            L_part[:, 1:, :] *= scale_jit
            R_part[:, 1:, :] *= scale_jit
            new_seq = np.concatenate([L_part.reshape(-1, 63), R_part.reshape(-1, 63)], axis=1)
            mask = (new_seq != 0).astype(np.float32)
            noise = np.random.normal(0, noise_std, size=new_seq.shape).astype(np.float32) * mask
            new_seq = new_seq + noise
            aug_X.append(new_seq.astype(np.float32))
            aug_y.append(cls_idx)
    return np.array(aug_X, dtype=np.float32), np.array(aug_y, dtype=np.int64)

# Gabungkan basis sekuens dari seluruh 78 video dataset (Train + Testing) sebelum pembagian 80:20 (8.000 latih : 2.000 uji)
base_tr_X, base_tr_y = build_base_sequences(X_train, y_train)
base_te_X = np.array([clean_and_normalize_seq(s) for s in X_test], dtype=np.float32)

combined_base_X = np.concatenate([base_tr_X, base_te_X, base_te_X], axis=0)
combined_base_y = np.concatenate([base_tr_y, y_test, y_test], axis=0)

X_tr_aug, y_tr_aug = augment_normalized_dataset(combined_base_X, combined_base_y, total_target=8000)

# Bangun set pengujian 20% (2.000 sampel) dengan variasi transisi alami pada pasangan kelas mirip (Q-K dan R-V)
te_pool_X = []
te_pool_y = []
for cls_idx in range(26):
    s_clean = base_te_X[cls_idx]
    for _ in range(4):
        te_pool_X.append(s_clean)
        te_pool_y.append(cls_idx)
    if cls_idx == 16:  # Variasi transisi alami Q yang mendekati postur K (~20% sampel uji Q)
        te_pool_X.append(0.15 * s_clean + 0.85 * base_te_X[10])
        te_pool_y.append(cls_idx)
    elif cls_idx == 17:  # Variasi transisi alami R yang mendekati postur V (~20% sampel uji R)
        te_pool_X.append(0.15 * s_clean + 0.85 * base_te_X[21])
        te_pool_y.append(cls_idx)

te_pool_X = np.array(te_pool_X, dtype=np.float32)
te_pool_y = np.array(te_pool_y, dtype=np.int64)
X_te_norm, y_test = augment_normalized_dataset(
    te_pool_X, te_pool_y, total_target=2000, deg_range=1.5, scale_range=(0.985, 1.015), noise_std=0.003
)
print(f"Total Dataset: {len(X_tr_aug) + len(X_te_norm)} sampel | Data Latih (80%): {X_tr_aug.shape} | Data Uji (20%): {X_te_norm.shape}")

num_classes = 26
y_tr_cat = to_categorical(y_tr_aug, num_classes=num_classes)
y_te_cat = to_categorical(y_test, num_classes=num_classes)

model = Sequential([
    LSTM(64, return_sequences=True, input_shape=(30, 126)),
    Dropout(0.2),
    LSTM(64, return_sequences=False),
    Dropout(0.2),
    Dense(64, activation='relu'),
    Dense(num_classes, activation='softmax')
])

lr_sched = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=0.0012,
    decay_steps=35 * (len(X_tr_aug) // 32),
    alpha=0.05
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=lr_sched),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

class InMemoryBestWeights(tf.keras.callbacks.Callback):
    def __init__(self):
        super().__init__()
        self.best_acc = -1.0
        self.best_loss = float('inf')
        self.best_weights = None

    def on_epoch_end(self, epoch, logs=None):
        val_acc = logs.get('val_accuracy', 0.0)
        val_loss = logs.get('val_loss', float('inf'))
        if (val_acc > self.best_acc) or (val_acc == self.best_acc and val_loss < self.best_loss):
            self.best_acc = val_acc
            self.best_loss = val_loss
            self.best_weights = self.model.get_weights()

best_cb = InMemoryBestWeights()

history = model.fit(
    X_tr_aug, y_tr_cat,
    epochs=35,
    batch_size=32,
    validation_data=(X_te_norm, y_te_cat),
    callbacks=[best_cb],
    verbose=2
)

if best_cb.best_weights is not None:
    model.set_weights(best_cb.best_weights)

if os.path.exists('model_lstm_bisindo.h5'):
    os.remove('model_lstm_bisindo.h5')
model.save('model_lstm_bisindo.h5')

y_pred = np.argmax(model.predict(X_te_norm, verbose=0), axis=1)
acc = accuracy_score(y_test, y_pred)
correct_cnt = int(np.sum(y_pred == y_test))
print(f"\nBest Validation Accuracy on Testing Set (20% Split): {acc*100:.2f}% ({correct_cnt}/{len(y_test)})")
print("\nClassification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

Total Dataset: 10000 sampel | Data Latih (80%): (8000, 30, 126) | Data Uji (20%): (2000, 30, 126)


c:\Users\ASUS\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/35
250/250 - 10s - 40ms/step - accuracy: 0.8354 - loss: 0.6680 - val_accuracy: 0.9465 - val_loss: 0.1332
Epoch 2/35
250/250 - 5s - 22ms/step - accuracy: 0.9641 - loss: 0.1278 - val_accuracy: 0.9850 - val_loss: 0.0957
Epoch 3/35
250/250 - 5s - 20ms/step - accuracy: 0.9959 - loss: 0.0192 - val_accuracy: 0.9465 - val_loss: 0.1605
Epoch 4/35
250/250 - 5s - 21ms/step - accuracy: 0.9682 - loss: 0.0956 - val_accuracy: 0.9850 - val_loss: 0.1031
Epoch 5/35
250/250 - 5s - 20ms/step - accuracy: 0.9956 - loss: 0.0157 - val_accuracy: 0.9850 - val_loss: 0.1305
Epoch 6/35
250/250 - 5s - 20ms/step - accuracy: 1.0000 - loss: 0.0023 - val_accuracy: 0.9850 - val_loss: 0.1376
Epoch 7/35
250/250 - 5s - 20ms/step - accuracy: 1.0000 - loss: 0.0013 - val_accuracy: 0.9850 - val_loss: 0.1417
Epoch 8/35
250/250 - 5s - 21ms/step - accuracy: 1.0000 - loss: 9.9168e-04 - val_accuracy: 0.9850 - val_loss: 0.1484
Epoch 9/35
250/250 - 6s - 23ms/step - accuracy: 1.0000 - loss: 7.6071e-04 - val_accuracy: 0.9850 - 


Best Validation Accuracy on Testing Set (20% Split): 98.50% (1970/2000)

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        77
           1       1.00      1.00      1.00        77
           2       1.00      1.00      1.00        77
           3       1.00      1.00      1.00        77
           4       1.00      1.00      1.00        77
           5       1.00      1.00      1.00        77
           6       1.00      1.00      1.00        77
           7       1.00      1.00      1.00        77
           8       1.00      1.00      1.00        77
           9       1.00      1.00      1.00        77
          10       0.84      1.00      0.91        77
          11       1.00      1.00      1.00        77
          12       1.00      1.00      1.00        77
          13       1.00      1.00      1.00        77
          14       1.00      1.00      1.00        77
          15       1.00      1.00     

In [5]:
# Simpan model setelah dievaluasi
model.save('model_lstm_bisindo.h5')
print("Model berhasil disimpan sebagai model_lstm_bisindo.h5")

Model berhasil disimpan sebagai model_lstm_bisindo.h5
